In [1]:
import pandas as pd
import json
import numpy as np
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

# 1. Load CSV data from local file
print("Loading Titanic dataset...")

# FIXED: Changed "data" to "Data" to match the folder you created
file_path = Path("Data/titanic.csv")

if not file_path.exists():
    raise FileNotFoundError(f"Dataset not found at: {file_path.resolve()}")

df = pd.read_csv(file_path)

print("✓ Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# 2. Calculate descriptive statistics
print("\nCalculating descriptive statistics...")
numerical_columns = df.select_dtypes(include=["number"]).columns.tolist()

stats_summary = {}
for col in numerical_columns:
    # FIXED: Wrapped in float() to prevent JSON serialization errors
    stats_summary[col] = {
        "mean": float(df[col].mean()) if pd.notnull(df[col].mean()) else 0.0,
        "median": float(df[col].median()) if pd.notnull(df[col].median()) else 0.0,
        "std_dev": float(df[col].std()) if pd.notnull(df[col].std()) else 0.0
    }

print("✓ Descriptive statistics calculated")

# 3. Identify and count missing values
print("\nChecking missing values...")
# FIXED: Ensure keys are strings and values are standard ints for JSON
missing_values = {str(k): int(v) for k, v in df.isnull().sum().items()}
total_missing = int(df.isnull().sum().sum())

for col, count in missing_values.items():
    print(f"{col}: {count}")

# 4. Clean missing data
print("\nCleaning missing data...")
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Cabin"] = df["Cabin"].fillna("Unknown")

# FIXED: Added a safe check in case 'Embarked' is completely empty
embarked_mode = df["Embarked"].mode()
df["Embarked"] = df["Embarked"].fillna(embarked_mode[0] if not embarked_mode.empty else "S")

print("✓ Missing values handled")

# 5. Feature engineering
print("\nPerforming feature engineering...")
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = df["FamilySize"].apply(lambda x: 1 if x == 1 else 0)
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
df["FarePerPerson"] = df["Fare"] / df["FamilySize"]

def age_group(age):
    if age < 18:
        return "Child"
    elif age < 60:
        return "Adult"
    else:
        return "Senior"

df["AgeGroup"] = df["Age"].apply(age_group)

print("✓ Engineered features created")

# 6. Analyze engineered features by survival
print("\nAnalyzing survival patterns...")
# FIXED: Cast Pandas/Numpy types to native python int/float for JSON saving
survival_by_isalone = {int(k): float(v) for k, v in df.groupby("IsAlone")["Survived"].mean().items()}
survival_by_agegroup = {str(k): float(v) for k, v in df.groupby("AgeGroup")["Survived"].mean().items()}
survival_by_title = {str(k): float(v) for k, v in df.groupby("Title")["Survived"].mean().sort_values(ascending=False).head(10).items()}
survival_by_familysize = {int(k): float(v) for k, v in df.groupby("FamilySize")["Survived"].mean().items()}

analysis_results = {
    "survival_by_isalone": survival_by_isalone,
    "survival_by_agegroup": survival_by_agegroup,
    "top_survival_by_title": survival_by_title,
    "survival_by_familysize": survival_by_familysize
}

print("✓ Survival analysis complete")

# 7. Export data to JSON using Python classes
@dataclass
class PassengerRecord:
    PassengerId: int
    Survived: int
    Pclass: int
    Name: str
    Sex: str
    Age: float
    SibSp: int
    Parch: int
    Ticket: str
    Fare: float
    Cabin: str
    Embarked: str
    FamilySize: int
    IsAlone: int
    Title: str
    FarePerPerson: float
    AgeGroup: str

@dataclass
class TitanicAnalysisExport:
    dataset_shape: Dict[str, int]
    missing_values: Dict[str, int]
    descriptive_statistics: Dict[str, Dict[str, float]]
    engineered_feature_analysis: Dict[str, Dict[Any, float]]
    passengers: List[Dict[str, Any]]

print("\nPreparing JSON export...")
passenger_records = []

for _, row in df.iterrows():
    passenger = PassengerRecord(
        PassengerId=int(row["PassengerId"]),
        Survived=int(row["Survived"]),
        Pclass=int(row["Pclass"]),
        Name=str(row["Name"]),
        Sex=str(row["Sex"]),
        Age=float(row["Age"]),
        SibSp=int(row["SibSp"]),
        Parch=int(row["Parch"]),
        Ticket=str(row["Ticket"]),
        Fare=float(row["Fare"]),
        Cabin=str(row["Cabin"]),
        Embarked=str(row["Embarked"]),
        FamilySize=int(row["FamilySize"]),
        IsAlone=int(row["IsAlone"]),
        Title=str(row["Title"]),
        FarePerPerson=float(row["FarePerPerson"]),
        AgeGroup=str(row["AgeGroup"])
    )
    passenger_records.append(asdict(passenger))

export_data = TitanicAnalysisExport(
    dataset_shape={"rows": int(df.shape[0]), "columns": int(df.shape[1])},
    missing_values=missing_values,
    descriptive_statistics=stats_summary,
    engineered_feature_analysis=analysis_results,
    passengers=passenger_records
)

json_output = asdict(export_data)
print("✓ JSON data prepared")

# 8. Validate JSON output
print("\nValidating JSON output...")
required_keys = [
    "dataset_shape",
    "missing_values",
    "descriptive_statistics",
    "engineered_feature_analysis",
    "passengers"
]

validation_passed = True

for key in required_keys:
    if key not in json_output:
        print(f"Missing key: {key}")
        validation_passed = False

if not isinstance(json_output["passengers"], list):
    print("Passengers should be a list")
    validation_passed = False

if len(json_output["passengers"]) == 0:
    print("Passengers list is empty")
    validation_passed = False

if validation_passed:
    print("✓ JSON validation passed")

# 9. Save files
print("\nSaving output files...")
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

csv_path = output_dir / "titanic_processed.csv"
json_path = output_dir / "titanic_analysis.json"

df.to_csv(csv_path, index=False)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_output, f, indent=4)

print("✓ Files saved successfully")
print(f"CSV saved to: {csv_path.resolve()}")
print(f"JSON saved to: {json_path.resolve()}")

# 10. Final summary
print("\n" + "=" * 50)
print("TITANIC ANALYSIS COMPLETE")
print("=" * 50)
print(f"Dataset shape: {df.shape}")
print(f"Total missing values before cleaning: {total_missing}")
print("\nSample survival analysis:")
print("Survival by IsAlone:", survival_by_isalone)
print("Survival by AgeGroup:", survival_by_agegroup)

Loading Titanic dataset...


FileNotFoundError: Dataset not found at: C:\Users\Adetunji\IRONHACK_LAB\Titanic_dataset\data\Data\titanic.csv